# CE541E08 — Unit 4 · Day 34 — Mapping and Replacing Codes

| **CO** | CO4 | **Topics** | map() · replace() · cut() · categorising data for analysis |
|---|---|---|---|

---

In [ ]:
student_name="Your Name"; roll="2024XXXXXX"
print(f"CE541E08 | Day 34 | {student_name} | {roll}")

### ▶ Cell 1 — pd.cut(): bin continuous data

In [ ]:
import pandas as pd, numpy as np

np.random.seed(2)
daily_flow = pd.Series(np.round(np.random.exponential(300,60),1),
                       index=pd.date_range('2024-07-01',periods=60,freq='D'),
                       name='Flow_m3s')

# Bin into flow regime categories
bins   = [0, 100, 300, 600, 1000, 5000]
labels = ['Very Low','Low','Moderate','High','Very High']
daily_flow_cat = pd.cut(daily_flow, bins=bins, labels=labels)

print("Flow value counts:")
print(daily_flow_cat.value_counts().sort_index())

# Combine into a DataFrame
df = pd.DataFrame({'Flow_m3s':daily_flow,'Category':daily_flow_cat})
print()
print("Mean flow per category:")
print(df.groupby('Category',observed=True)['Flow_m3s'].mean().round(1))
# --- INSTRUCTOR NOTE ---
# pd.cut() bins continuous values into categories — like IMD thresholds
# observed=True suppresses warning about unused categories

### ▶ Cell 2 — IMD rainfall binning

In [ ]:
import pandas as pd, numpy as np

np.random.seed(10)
daily_rain = pd.Series(np.round(np.random.exponential(20,122),1),
                       index=pd.date_range('2024-06-01',periods=122,freq='D'),
                       name='Rainfall_mm')

bins   = [-0.1, 0, 15.5, 64.4, 115.5, 204.4, 1000]
labels = ['No rain','Light','Moderate','Heavy','Very Heavy','Extremely Heavy']
categories = pd.cut(daily_rain, bins=bins, labels=labels)
counts = categories.value_counts().sort_index()
total  = len(daily_rain)
print("IMD Rainfall Classification — Monsoon Season 2024")
print(f"{'Category':<22} {'Days':>6} {'%':>7}")
print("-"*37)
for cat, cnt in counts.items():
    print(f"{cat:<22} {cnt:>6} {cnt/total*100:>6.1f}%")
print(f"{'TOTAL':<22} {total:>6}")

### ▶ Cell 3 — replace() for code cleanup

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Date'     : ['2024-07-01','2024-07-02','2024-07-03','2024-07-04'],
    'Flow_m3s' : [234.5, 678.9, -999.0, 890.2],
    'QA_code'  : ['G','G','M','H'],
    'Soil_type': ['SL','CL','C','SL'],
})

# Replace -999 with NaN
df['Flow_m3s'] = df['Flow_m3s'].replace(-999.0, float('nan'))

# Replace codes with descriptions
df['QA_Flag']  = df['QA_code'].replace({'G':'GOOD','M':'MISSING','H':'HIGH'})
df['Soil_name']= df['Soil_type'].replace({'SL':'Sandy Loam','CL':'Clay Loam','C':'Clay'})

print(df[['Date','Flow_m3s','QA_Flag','Soil_name']])
print(f"Valid flow mean: {df['Flow_m3s'].mean():.2f} m3/s")

### ▶ Cell 4 — Crosstab: flow regime vs season

In [ ]:
import pandas as pd, numpy as np

np.random.seed(5)
dates = pd.date_range('2022-01-01','2024-12-31',freq='D')
base  = np.where((dates.month>=6)&(dates.month<=9),400,80)
flow  = np.round(np.maximum(base+np.random.normal(0,base*0.3,len(dates)),5),1)
df    = pd.DataFrame({'Flow_m3s':flow},index=dates)
df['Month']   = df.index.month
df['Year']    = df.index.year
df['Season']  = df['Month'].apply(lambda m:'Monsoon' if 6<=m<=9 else 'Non-monsoon')
df['Regime']  = pd.cut(df['Flow_m3s'],bins=[0,100,300,600,5000],labels=['Low','Moderate','High','Very High'])

ct = pd.crosstab(df['Regime'],df['Season'])
print("Flow regime vs Season (days count):")
print(ct)
print()
pct = ct.div(ct.sum())*100
print("Percentage distribution:")
print(pct.round(1))

---
## Day 34 Assignment
Load a DataFrame of 30 days of daily data (Date, Rainfall_mm, Temperature_C, Flow_m3s).
1. Use pd.cut() to classify rainfall into IMD categories
2. Use pd.cut() to classify flow into Low/Medium/High/Flood
3. Crosstab: rainfall category vs flow category
4. Which rainfall category most often leads to High/Flood flow?

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np
np.random.seed(7)
n=30
df=pd.DataFrame({'Date':pd.date_range('2024-07-01',periods=n,freq='D'),
                 'Rainfall_mm':np.round(np.random.exponential(35,n),1),
                 'Flow_m3s':np.round(np.random.exponential(300,n),1)})
rain_bins=[-0.1,0,15.5,64.4,115.5,1000]
rain_labs=['No rain','Light','Moderate','Heavy','V.Heavy']
flow_bins=[0,100,400,800,9999]
flow_labs=['Low','Medium','High','Flood']
df['Rain_cat'] = pd.cut(df['Rainfall_mm'],bins=rain_bins,labels=rain_labs)
df['Flow_cat'] = pd.cut(df['Flow_m3s'],bins=flow_bins,labels=flow_labs)
print(df[['Date','Rainfall_mm','Rain_cat','Flow_m3s','Flow_cat']].head(10))
ct = pd.crosstab(df['Rain_cat'],df['Flow_cat'])
print("Crosstab:"); print(ct)

---
- [ ] Upload: `Unit4_Pandas/CE541E08_U4_Day34.ipynb`

*CE541E08 · Civil Engineering · Christ University*